# Mushroom Classifier
This model's goal is to classify images of mushrooms and determine whether they are poisonous or safe.  
The model was trained with an AMD GPU, which caused a fair amount of issues and forced some architecture changes.

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import keras
from keras import layers

import numpy as np
import matplotlib.pyplot as plt

from keras import backend as K
import tensorflow as tf

# Verification block
print(f"Keras version: {keras.__version__}")
print(f"TensorFlow version: {tf.__version__}")

# This will list your DirectML adapters (GPU:0 and GPU:1)
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Detected: {gpus}")

C:\Users\Omistaja\anaconda3\envs\tf_directml\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Keras version: 2.10.0
TensorFlow version: 2.10.0
GPUs Detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


Performing data augmentation while the Jupyter server was running always crashed the kernel during training phase.  
To fix this, the entire dataset is augmented at once and saved as a new dataset.  
This means that the images are already augmented when loaded as a dataset.

In [2]:
from sklearn.model_selection import train_test_split
batch_size_n = 64
img_size = (224, 224)

# Load images
dataset = keras.utils.image_dataset_from_directory(
    '../datasets/mushroom_augmented',
    batch_size=batch_size_n, 
    image_size=img_size,
    crop_to_aspect_ratio=True,
    labels='inferred',
    label_mode='categorical',
    shuffle=True,
    seed=123)

num_classes = len(dataset.class_names)

Found 17461 files belonging to 9 classes.


The total dataset is now much larger than the original, because of the preprocessed images.

In [3]:
# Split dataset into validation and training
dataset_size = len(dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)

train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size).take(val_size)
test_dataset = dataset.skip(train_size + val_size)

In [4]:
# Prefetch datasets to improve performance
import tensorflow as tf
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [5]:
from keras import Sequential
from keras import layers
from keras.optimizers import Adam

Because of how older versions of tensorflow work, data augmentation MUST be done outside of the training.  
CPU bottleneck kills performance, making training take ~20x longer.

In [6]:
model = Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./255),
    
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.GlobalAveragePooling2D(), # Or layers.Flatten() for more parameters
    
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 224, 224, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 112, 112, 32)     0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 112, 112, 64)      18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 56, 56, 64)       0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 56, 56, 128)       7

In [7]:
model.compile(optimizer=Adam(learning_rate = 0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
from keras.callbacks import EarlyStopping

history = model.fit(
    train_dataset,
    batch_size=batch_size_n,
    epochs=64,
    validation_data=val_dataset,
    callbacks=[
        EarlyStopping(patience=5, restore_best_weights=True)
    ]
)

Epoch 1/64
191/191 [==============================] - 14s 65ms/step - loss: 2.0817 - accuracy: 0.2052 - val_loss: 2.0574 - val_accuracy: 0.2164
Epoch 2/64
191/191 [==============================] - 6s 29ms/step - loss: 2.0201 - accuracy: 0.2179 - val_loss: 1.9687 - val_accuracy: 0.2406
Epoch 3/64
191/191 [==============================] - 6s 29ms/step - loss: 1.9572 - accuracy: 0.2434 - val_loss: 1.9184 - val_accuracy: 0.2945
Epoch 4/64
191/191 [==============================] - 6s 29ms/step - loss: 1.9140 - accuracy: 0.2749 - val_loss: 1.8768 - val_accuracy: 0.3156
Epoch 5/64
191/191 [==============================] - 6s 29ms/step - loss: 1.8802 - accuracy: 0.2972 - val_loss: 1.8427 - val_accuracy: 0.3246
Epoch 6/64
191/191 [==============================] - 6s 29ms/step - loss: 1.8547 - accuracy: 0.3148 - val_loss: 1.8221 - val_accuracy: 0.3383
Epoch 7/64
191/191 [==============================] - 6s 29ms/step - loss: 1.8319 - accuracy: 0.3255 - val_loss: 1.8026 - val_accuracy: 0.347

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(test_dataset, verbose=0)
print(f"The model's test accuracy is {acc}")